Classification script
---------------------
This script classifies an input .csv with LinkedIn scrapes according to 10 LSEG ESG subcategories using a LLM. It's used for both ground truth sample and final entire LinkedIn dataset. Final configuration is the following:
- MODE = 1
- EXAMPLES = True
- REASONING = False
- SELF_CORRECTION = False
- ROLE_NR = 5
- MODEL = "GPT5n"

In [1]:
import pandas as pd
from openai import OpenAI
import time
import json
import os
import sys
from tqdm import tqdm
from dotenv import load_dotenv

configuration:

In [ ]:
# Load the API key from your specific .env file / put this file in the parent directory ('LLM Analyses' folder)
load_dotenv(os.path.join("..", "API key.env"))
API_KEY = os.getenv("OPENAI_API_KEY")
if API_KEY: print("API_KEY loaded succesfully")

# Prompt Technique Toggles
EXAMPLES = False    # Set to True to include 3-shot examples input
REASONING = True   # Set to True for Chain-of-thought reasoning output
SELF_CORRECTION = False     # Set to True to make the LLM correct its previous output, only do this if the configuration output already exists
ROLE_NR = 4     # Choose the system prompt (role): "0" means leaving out the role in system prompt
ROLE = {
        0: "no role",
        1: "an expert in ESG investment",   # Shimamura et al., 2025    - readability
        2: "a professor of linguistics",   # Shimamura et al., 2025    - readability
        3: "an average worker",    # Shimamura et al., 2025    - readability
        4: "an ESG auditor",   # Jakubczak et al., 2025    - classification
        5: "a sustainability expert",  # Ghasemi, 2024     - classification
        6: "a trained research assistant"   # Tripp, 2024   - classification
         }[ROLE_NR]

# Model configuration
MODEL = "GPT5n"
MODEL_ID = {"GPT5.1": "gpt-5.1", "GPT5": "gpt-5", "GPT5n": "gpt-5-nano"}[MODEL]  # This script works exclusively for OpenAI models
TEMPERATURE = 1     # 0 for strictly deterministic, however, GPT 5 only allows 1 (as it's a reasoning model)

# Files configuration
DATASET = ["ground truth", "final dataset"][0]  # Executes classification on either the final dataset or the ground truth sample
ROOT = "--FILL IN YOUR ROOT FOLDER--"  # root project folder
INPUT_DIR = {"final dataset": f"{ROOT}/Data Collection/LinkedIn", "ground truth": f"{ROOT}/Ground Truth/Samples"}[DATASET]
TXT_FILES_DIR = "."
OUTPUT_DIR = {"final dataset": ".", "ground truth": "Ground Truth sample - clsf/Classifications"}[DATASET]

CATEGORY_FILE = "Classification categories.txt"
OUTPUT_FILE = f"Classification_{MODEL}_role{ROLE_NR}{"_examples" if EXAMPLES else ""}{"_reasoning" if REASONING else ""}.csv"
INPUT_FILE = {"final dataset": "LinkedIn 2025-26 Dataset - clsf.csv", "ground truth": "LinkedIn Ground Truth Sample - clsf large.csv"}[DATASET]
EXAMPLES_FILE = f"Classification examples.txt"

# Script Parameters
MAX_POSTS = None                        # "None" for full analysis
POST_START = 0                       # Index of the post where analysis should start (0 is first post)
CHECKPOINT_INTERVAL = 300              # Save after x posts
SLEEP_TIME = 1                        # Break (in seconds) between API calls
RETRIES = 3                             # Amount of retries for 500 / 503 errors

# The 10 official LSEG subcategories
SUBCATEGORIES = [
    "Resource Use", "Emissions", "Innovation", 
    "Workforce", "Human Rights", "Community", "Product Responsibility", 
    "Management", "Shareholders", "CSR Strategy"
]

# Mapping from subcategory to parent category
PARENT_MAP = {
    "Resource Use": "E", "Emissions": "E", "Innovation": "E",
    "Workforce": "S", "Human Rights": "S", "Community": "S", "Product Responsibility": "S",
    "Management": "G", "Shareholders": "G", "CSR Strategy": "G",
    "None": "N"
}

main algorithm:

In [ ]:
client = OpenAI(api_key=API_KEY)

def load_definitions(filepath):
    """Loads ESG definitions from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Definition file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def load_examples(filepath):
    """
    Loads few-shot examples from text file. Raises error if file is missing. 
    Otherwise deletes reasoning in the examples if disabled.
    """
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Examples file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
        
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        full_text =  f.read()

    # Opened file, moving on to filter out unwanted lines
    lines = full_text.splitlines()
    filtered_lines = []

    for line in lines:
        if not REASONING and '"reasoning":' in line:
            continue
        filtered_lines.append(line)

    # Clean up overhanging commas before closing brackets
    clean_lines = []
    total_lines = len(filtered_lines)
    for i in range(total_lines):
        current_line = filtered_lines[i]
        # Check if there is a next line, and if that next line contains a closing brace '}'
        if i + 1 < total_lines and "}" in filtered_lines[i + 1]:
            # If yes, the current line is the new final item; strip its trailing comma!
            current_line = current_line.rstrip(",")     
        clean_lines.append(current_line)
        
    # Rejoin all processed lines back into a single string
    return "\n".join(clean_lines)

def get_previous_output_string(row):
    """
    Reconstructs a readable JSON-like string from the dataframe row. 
    Only used in case of Self-Correction to integrate previous classifications in prompt.
    """
    data = {}
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        sub_dict = {"active": row.get(f"{clean_sub}_active", "No")}
        if REASONING:
            sub_dict["reasoning"] = row.get(f"{clean_sub}_reasoning", "")
        data[clean_sub] = sub_dict
    print(json.dumps(data, indent=2))
    return json.dumps(data, indent=2)

def build_prompt(post_text, definitions, examples, pr_clsf):
    """Dynamically builds the prompt based on configuration."""
    
    # Constructing the dynamic JSON schema description
    schema_fields = []
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        details = ['"active": "Yes/No"']
        if REASONING: details.append('"reasoning": "string"')
        schema_fields.append(f'"{clean_sub}": {{{", ".join(details)}}}')

    prompt = f"""{"""
Your task is to produce a verified classification for the post below. 
You have already classified the LinkedIn post and are now reviewing your previous output.
1. Review your initial analysis critically. Ensure you haven't implied categories, only explicit evidence counts.
2. If the initial analysis is correct, your verified output should match it exactly. If you find an error, your verified output must reflect the correction.
You must output the full JSON structure for every post.

Initial prompt:
""" if SELF_CORRECTION else ""}Your task is to analyze LinkedIn posts and classify them into 10 specific LSEG ESG subcategories.

Definitions of the LSEG ESG subcategories:
{definitions}

Instructions:
1. Determine if the article contains explicit thematic evidence of one or more of the 10 LSEG ESG subcategories. 
You are identifying the presence of a topic, regardless of whether the information is positive, negative, or neutral. 
Only classify based on explicit evidence. NEVER derive or imply categories.
2. If there is absolutely no concrete evidence for the subcategory, set the "active" status for the subcategory to "No". 
Even if there is no evidence for any of the categories, you MUST {"provide a justification for each subcategory why it's not relevant and" if REASONING else ""} set the "active" status for all 10 subcategories to "No". 
Never provide an empty output.
{ "3. Provide a detailed reasoning before executing every active subcategory classification." if REASONING else "" }
{ f"Examples:\n{examples}" if EXAMPLES else ""}

Strict Output Format (JSON):
{{
    {", ".join(schema_fields)}
}}

Input (LinkedIn Post):
{post_text}{f"\n\nYour initial analysis:\n{pr_clsf}" if SELF_CORRECTION else ""}"""
    
    return prompt

def get_response_schema():
    """Defines the JSON schema in OpenAI's expected format."""
    properties = {}
    
    sub_props = {
        "active": {"type": "string", "enum": ["Yes", "No"]}
    }
    if REASONING:
        sub_props["reasoning"] = {"type": "string"}

    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        properties[clean_sub] = {
            "type": "object",
            "properties": sub_props,
            "required": list(sub_props.keys()),
            "additionalProperties": False
        }
    
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "esg_classification",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": properties,
                "required": list(properties.keys()),
                "additionalProperties": False
            }
        }
    }

def classify_post(post_text, definitions, examples, retries, pr_clsf):
    """Sends prompt to GPT-5 using Structured Outputs."""
    prompt = build_prompt(post_text, definitions, examples, pr_clsf)

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {ROLE}."},
                    {"role": "user", "content": prompt}
                ]  if ROLE_NR else [{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                response_format=get_response_schema()
            )
            # OpenAI returns the string in .message.content
            return json.loads(response.choices[0].message.content)
            
        except Exception as e:
            if "503" in str(e) or "500" in str(e) or "rate_limit" in str(e).lower():
                print(f"\n[Attempt {attempt + 1}] API Error. Retrying...")
                time.sleep(10 * (attempt + 1))
                continue
            return {"error": str(e)}
            
    return {"error": "Maximum retries reached."}

def main():
    # Continue with the previous output if output has already been created for this configuration
    if os.path.exists(os.path.join(OUTPUT_DIR, OUTPUT_FILE)):
        input_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
        print("Continuing with previous output...")
    else:
        input_path = os.path.join(INPUT_DIR, INPUT_FILE)

    # Add "_selfc" to the output file if Self Correction is enabled
    output_file = OUTPUT_FILE.replace(".csv", "_selfc.csv") if SELF_CORRECTION else OUTPUT_FILE

    # 1. Verification of required files
    try:
        definitions = load_definitions(os.path.join(TXT_FILES_DIR, CATEGORY_FILE))
        examples = load_examples(os.path.join(TXT_FILES_DIR, EXAMPLES_FILE))
    except FileNotFoundError:
        sys.exit(1)

    if not os.path.exists(input_path):
        print(f"Error: Input file not found in {input_path}.")
        return

    # 2. Load Data
    df = pd.read_csv(input_path)
    print("Input file opened succesfully")
    # Expected columns: Company Name, Date, URL, Text
    text_col = "Text" 
    # If Self Correction is enabled: make use of original dataframe variable for reference classifications
    df_orig = df.copy() if SELF_CORRECTION else None

    # 3. Initialize Columns
    # If Self Correction is enabled: wipe all existing classification columns
    # Ensure main category columns exist
    for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
        if cat not in df.columns: df[cat] = None
        elif SELF_CORRECTION: df[cat] = None
        
    # Ensure subcategory columns exist
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        cols = [f"{clean_sub}_active"]
        if REASONING: cols.append(f"{clean_sub}_reasoning")
        
        for col in cols:
            if col not in df.columns: df[col] = None
            elif SELF_CORRECTION: df[col] = None

    # Calculate range
    end_index = len(df)
    if MAX_POSTS is not None:
        end_index = min(POST_START + MAX_POSTS, len(df))

    print(f"Processing posts from index {POST_START} to {end_index}...")

    # 4. Processing Loop
    print("Running", output_file.replace("_", " "))
    for index in tqdm(range(POST_START, end_index), desc="Classifying"):
        # Skip if already successfully processed
        if pd.notna(df.at[index, 'Cat_E']) and "ERROR" not in str(df.at[index, 'Cat_N']):
            continue

        post_content = df.at[index, text_col]
        # Get previous classification if Self Correction is enabled
        pr_clsf = get_previous_output_string(df_orig.iloc[index]) if SELF_CORRECTION else None

        result = classify_post(post_content, definitions, examples, RETRIES, pr_clsf)

        if "error" in result:
            df.at[index, 'Cat_N'] = "ERROR: " + result["error"]
        else:
            # Map Main Categories, base is "No"
            for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
                df.at[index, cat] = "No"
            any_sub_active = False

            # Map Subcategories directly
            for sub in SUBCATEGORIES:
                clean_sub = sub.replace(" ", "_")
                sub_data = result.get(clean_sub, {})

                is_active = sub_data.get("active", "No")
                df.at[index, f"{clean_sub}_active"] = is_active

                if is_active == "Yes":
                    any_sub_active = True
                    # Zoek de hoofdletter op via je PARENT_MAP (E, S, of G)
                    parent_cat = PARENT_MAP.get(sub)
                    if parent_cat:
                        df.at[index, f"Cat_{parent_cat}"] = "Yes"

                if REASONING:
                    df.at[index, f"{clean_sub}_reasoning"] = sub_data.get("reasoning", "")

            if not any_sub_active: df.at[index, "Cat_N"] = "Yes"
            
        # Save every X posts to prevent data loss
        if (index + 1) % CHECKPOINT_INTERVAL == 0:
            df.to_csv(os.path.join(OUTPUT_DIR, output_file), index=False, encoding='utf-8-sig')
            print(f" Checkpoint: Saved up to index {index}")

        time.sleep(SLEEP_TIME)

    # Final Save
    df.to_csv(os.path.join(OUTPUT_DIR, output_file), index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {output_file}")



if __name__ == "__main__":
    main()